In [73]:
import os,sys, re, json, nest_asyncio, asyncio, numpy as np, pandas as pd

SRC_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

from dotenv import load_dotenv

from google import genai
from google.genai import types, Client
from google.genai.types import EmbedContentConfig
from google.cloud import secretmanager, storage,aiplatform

from typing import List, Dict, Any, Union, Tuple
from pydantic import BaseModel, Field

import matplotlib.pyplot as plt
import seaborn as sns

from langchain_core.documents import Document
from langchain.embeddings.base import Embeddings
from sklearn.metrics.pairwise import cosine_similarity
from langchain.chains.summarize import load_summarize_chain
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from pydantic import BaseModel, Field
from langchain_google_vertexai import ChatVertexAI
from langchain_core.documents import Document
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from copy import deepcopy
import spacy
import nltk

from spacy.lang.en import English
import subprocess
import csv
import shutil
import pathlib
import os
import networkx as nx
import tiktoken

from langchain_community.llms.mlx_pipeline import MLXPipeline
from langchain_community.chat_models.mlx import ChatMLX
from mlx_lm import load, generate

from schemas.keywords import KeywordExtractionConfig
from document_processing.keyword_annotator import BM25KeywordAnnotator, TFIDFKeywordAnnotator, QueryProcessor, KeyBertAnnotator,ENGLISH_STOP_WORDS,NLTKKeywordAnnotator
from document_processing import TextDirectoryLoader, ChunkedTextDirectoryLoader
from indexing.inverted_index import InvertedIndex
from text_splitters import CustomTokenSplitter, CustomSemanticChunker, SpectralSegmentationChunker 

load_dotenv()

LLAMA_PARSE_API_KEY = os.environ.get("LLAMA_PARSE_API_KEY")
HF_TOKEN = os.environ.get("HUGGINGFACEHUB_API_TOKEN")
JAR_PATH = os.environ.get("JAR_PATH")

PROJECT_ID = os.environ.get("PROJECT_ID")
LOCATION = os.environ.get("LOCATION")
aiplatform.init(project=PROJECT_ID, location="global") 


embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-l6-v2")
emb = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-l6-v2")

bm25_annotator = BM25KeywordAnnotator({
    "score_threshold": 1.2,
    "extra_stopwords": {"page", "pages", "figure", "copyright", "©"},
    "max_keywords": 20,
})

spectral_bm25_chunker = ChunkedTextDirectoryLoader(
    #directory="../temp/test-data",
    directory="../Data/parsed",
    chunker = SpectralSegmentationChunker(
        embeddings=emb,
        chunk_size=4_000,            # token budget for *final* chunks
        window_k=8,                  # neighbourhood size for similarity graph
        min_words=40,                # enforce a sensible pre-chunk length
        keyword_annotator=bm25_annotator, # None if you don’t need per-chunk keywords
    ),
) 

all_chunks_bm25 = spectral_bm25_chunker.load()

def tok_len(text: str, model="cl100k_base") -> int:
    return len(tiktoken.get_encoding(model).encode(text))



Processing file: motherboard-manual_cleaned.txt

Processing file: aldi-document-2_cleaned.txt

Processing file: attention-is-all-you-need_cleaned.txt

Processing file: financial-report_cleaned.txt

Processing file: rental-agreement_cleaned.txt

Processing file: aldi-document-1_cleaned.txt


In [74]:
class ReVerbRelationshipExtractor:
    def __init__(self, jar_path=JAR_PATH, java_mem="512m", conf_thresh=0.0):
        if not shutil.which("java"):
            raise RuntimeError("Java runtime not found on PATH")
        if not pathlib.Path(jar_path).exists():
            raise FileNotFoundError(jar_path)
        self.jar_path = jar_path
        self.java_mem = java_mem
        self.conf_thresh = conf_thresh

    def reverb_extract(self, sentence:str) -> List[Tuple[str, str, str, float]]:
        sentence = (sentence.rstrip() + "\n")

        cmd = ["java", f"-Xmx{self.java_mem}", "-jar", self.jar_path, "-q"]
        proc = subprocess.run(cmd, input=sentence,
                            capture_output=True, text=True)

        if proc.returncode:
            raise RuntimeError(proc.stderr)

        triples = []
        for row in csv.reader(proc.stdout.splitlines(), delimiter='\t'):
            if len(row) < 12:          # malformed
                continue
            arg1, rel, arg2 = row[2:5]         # columns 2-4
            conf = float(row[11])              # column 11
            if conf >= self.conf_thresh:
                triples.append((arg1, rel, arg2, conf))
        return triples

    def extract(self, sentences: List[str]) -> List[Tuple[str, str, str, float]]:
        return [self.reverb_extract(sentence) for sentence in sentences]

class PostChunkingProcessor:
    def __init__(self):
        pass

    def normalise(self,ent: str) -> str:
        ent = ent.lower()
        ent = re.sub(r'[^\w\s]', '', ent)   # strip punctuation
        ent = re.sub(r'\b(the|a|an|this|these|those|that)\b', '', ent)
        ent = ' '.join(ent.split())
        return ent


    def keywords_ner(self,file_copy: List[Document], annotator: Union[NLTKKeywordAnnotator], relationship_extractor: Union[ReVerbRelationshipExtractor],nlp:Union[English]) -> List[Dict[str, Any]]:
        file = deepcopy(file_copy)
        full_text = '\n'.join([i.page_content for i in file])
        for chunk in file:
            text = chunk.page_content
            chunk_keywords = annotator(text)
            chunk.metadata['nlp_keywords'] = chunk_keywords
        
        relations_tuple = relationship_extractor.reverb_extract(full_text)

        cleaned_tuples = []
        for s, r, o, conf in relations_tuple:           # your 60 tuples
            if conf < 0.05:                     # drop very weak edges
                continue
            cleaned_tuples.append((self.normalise(s),
                            nlp(r).text.strip().lower(),   # lemma-ish
                            self.normalise(o),
                            conf))
        # optional: de-duplicate (max conf wins)
        uniq = {}
        for s,r,o,c in cleaned_tuples:
            key = (s,r,o)
            if c > uniq.get(key,0):
                uniq[key] = c
        triples = [(s,r,o,c) for (s,r,o),c in uniq.items()]
        return file,triples
    
class PostProcessingDocuments:
    def __init__(self, embeddings: Embeddings, documents: List[Document]):
        self.embeddings = embeddings
        self.documents = documents

    def chunks_cosine_similarity_matrix(self) -> np.ndarray:
        return cosine_similarity([self.embeddings.embed_query(chunk.page_content) for chunk in self.documents])
    
    def laplacian_matrix(self) -> np.ndarray:
        cos_sim_matrix = self.chunks_cosine_similarity_matrix()
        n_chunks = cos_sim_matrix.shape[0]
        W = cos_sim_matrix - np.eye(n_chunks)  # remove self-similarity
        D = np.diag(np.sum(W, axis=1))
        laplacian = D - W
        return laplacian

    def eigenvalues_eigenvectors(self) -> Tuple[np.ndarray, np.ndarray]:
        laplacian = self.laplacian_matrix()
        eigenvalues, eigenvectors = np.linalg.eig(laplacian)
        # sort eigen values and vectors based on eigenvalues
        idx = np.argsort(eigenvalues)
        eigenvalues = eigenvalues[idx]
        eigenvectors = eigenvectors[:, idx]
        return eigenvalues, eigenvectors


In [76]:
token_chunker = ChunkedTextDirectoryLoader(
    #directory="../temp/test-data",
    directory="../Data/parsed",
    chunker = CustomTokenSplitter(
        chunk_size=4_000,            
        chunk_overlap=400,            # token overlap between chunks
        keyword_annotator=bm25_annotator, # None if you don’t need per-chunk keywords
    ),
) 

all_chunks_bm25_token = token_chunker.load()


Processing file: motherboard-manual_cleaned.txt

Processing file: aldi-document-2_cleaned.txt

Processing file: attention-is-all-you-need_cleaned.txt

Processing file: financial-report_cleaned.txt

Processing file: rental-agreement_cleaned.txt

Processing file: aldi-document-1_cleaned.txt


In [77]:
#file_name = 'rental-agreement_cleaned.txt'
file_name = 'attention-is-all-you-need_cleaned.txt'
LOADER = all_chunks_bm25_token
full_text = "\n".join([text.page_content for text in LOADER[file_name] if text.page_content.strip()])
text_chunks = [text.page_content for text in LOADER[file_name] if text.page_content.strip()]
[tok_len(chunk) for chunk in text_chunks]

[3235, 2966, 3989, 400]

In [10]:
reverb_graph_extractor = ReVerbRelationshipExtractor(jar_path=JAR_PATH, java_mem="512m", conf_thresh=0.0)
nlp = spacy.load("en_core_web_sm", disable=["parser","ner"])
nltk_kp = NLTKKeywordAnnotator(processing_tags=[], non_processing_tags=['FW', 'NNP', 'NNPS'],stop_words=ENGLISH_STOP_WORDS) 


PCP = PostChunkingProcessor()
file,additional_data = PCP.keywords_ner(file_copy=LOADER[file_name], 
    annotator=nltk_kp,
    relationship_extractor=reverb_graph_extractor,
    nlp=nlp) 

PPD = PostProcessingDocuments(embeddings=emb, documents=LOADER[file_name])
eigenvalues, eigenvectors = PPD.eigenvalues_eigenvectors()

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [17]:
from langchain_core.prompts import PromptTemplate

summary_template = """YOu are a professional summarizer. Without making up any information, summarize the fallowing text that captures most important information in the text.
Summarize the fallowing text. Also indentify important relations that exist between entities in the text.
{page_content}

Summary:
"""

SUMMARY_TEMPLATE = PromptTemplate(
    input_variables=["page_content"],
    template=summary_template)

small_llm = ChatOllama(model="smollm2:360m",temperature=0)
mlx_llm = MLXPipeline.from_model_id(
    "mlx-community/Llama-3.2-1B-Instruct-4bit",
    #"mlx-community/gemma-3n-E2B-it-lm-4bit",
    #"mlx-community/Qwen3-0.6B-8bit",
    #"mlx-community/SmolLM2-360M-Instruct",
    pipeline_kwargs={"temp": 0.0,"max_tokens":512},
)

chat_model = ChatMLX(llm=mlx_llm)
summary_chain_mlx = load_summarize_chain(mlx_llm, chain_type="stuff")
summary_chain_ollama = load_summarize_chain(small_llm, chain_type="stuff")

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

In [18]:
[len(i.split(' ')) for i in text_chunks]

[2187, 2248, 2265, 286]

In [19]:
[mlx_llm.invoke(SUMMARY_TEMPLATE.format(page_content=text.page_content)) for text in LOADER[file_name]]

['The Transformer is a simple neural network architecture that uses self-attention mechanisms to draw global dependencies between input and output. It is based on the Transformer model architecture, which was proposed by Vaswani et al. [13]. The Transformer architecture is designed to be parallelizable and efficient, with a reduced number of operations compared to traditional recurrent neural networks. The Transformer architecture is also designed to be more efficient in terms of memory constraints, as it can be trained on a single GPU for a short period of time. The Transformer architecture is also designed to be more efficient in terms of training time, as it can be trained on a single GPU for a short period of time. The Transformer architecture is also designed to be more efficient in terms of memory usage, as it can be trained on a single GPU for a short period of time.\n\nImportant relations:\nThe Transformer architecture is based on the Transformer model architecture, which was p

In [14]:
summary_chain_mlx.invoke(LOADER[file_name])

{'input_documents': [Document(metadata={'source': 'attention-is-all-you-need_cleaned.txt', 'page_number': 1, 'pages': [1, 2, 3, 4, 5], 'keywords': ['add', 'additive', 'aidan', 'align', 'allow', 'applications', 'ashish', 'assign', 'attend', 'attribution', 'aug', 'bleu', 'boundary', 'brain', 'bytenet', 'codebase', 'come', 'compatibility', 'compose', 'computation']}, page_content='arXiv:1706.03762v7 [cs.CL] 2 Aug 2023\n\nProvided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works.\n\n# Attention Is All You Need\n\nAshish Vaswani* Google Brain avaswani@google.com\n\nNoam Shazeer* Google Brain noam@google.com\n\nNiki Parmar* Google Research nikip@google.com\n\nJakob Uszkoreit* Google Research usz@google.com\n\nLlion Jones* Google Research llion@google.com\n\nAidan N. Gomez* † University of Toronto aidan@cs.toronto.edu\n\nŁukasz Kaiser* Google Brain lukaszkaiser@google.com\n\nIlli

In [27]:
from summa.summarizer import summarize
from summa import keywords

In [35]:
summarize(full_text, words=100)

'To the best of our knowledge, however, the Transformer is the first transduction model relying entirely on self-attention to compute representations of its input and output without using sequence- aligned RNNs or convolution.\nEven with k = n, however, the complexity of a separable convolution is equal to the combination of a self-attention layer and a point-wise feed-forward layer, the approach we take in our model.\nIn this work, we presented the Transformer, the first sequence transduction model based entirely on attention, replacing the recurrent layers most commonly used in encoder-decoder architectures with multi-headed self-attention.'

In [39]:
keywords.keywords(full_text,split=True, scores=True)

[('models', 0.2753885578695895),
 ('model', 0.2753885578695895),
 ('modeling', 0.2753885578695895),
 ('attention', 0.27478055750752745),
 ('attentions', 0.27478055750752745),
 ('learn', 0.16629852181060514),
 ('learning', 0.16629852181060514),
 ('learned', 0.16629852181060514),
 ('learns', 0.16629852181060514),
 ('arxiv', 0.16550564240526447),
 ('layer', 0.15630806234310382),
 ('positions', 0.14272411450445413),
 ('positional', 0.14272411450445413),
 ('train', 0.13039626021813036),
 ('training', 0.13039626021813036),
 ('trained', 0.13039626021813036),
 ('sequence', 0.12546328701715825),
 ('sequences', 0.12546328701715825),
 ('position representation', 0.12473054468898212),
 ('k', 0.12447707932533952),
 ('computing', 0.11697157398558142),
 ('compute', 0.11697157398558142),
 ('computed', 0.11697157398558142),
 ('computes', 0.11697157398558142),
 ('representations', 0.1067369748735101),
 ('google', 0.1049267954548179),
 ('n', 0.10248889869814712),
 ('long', 0.09832337627716534),
 ('connec

In [44]:
[word for word, score in keywords.keywords(full_text,split=True, scores=True) if score > 0.1]

['models',
 'model',
 'modeling',
 'attention',
 'attentions',
 'learn',
 'learning',
 'learned',
 'learns',
 'arxiv',
 'layer',
 'positions',
 'positional',
 'train',
 'training',
 'trained',
 'sequence',
 'sequences',
 'position representation',
 'k',
 'computing',
 'compute',
 'computed',
 'computes',
 'representations',
 'google',
 'n']

In [52]:
from transformers import pipeline
classifier = pipeline("zero-shot-classification", model="MoritzLaurer/DeBERTa-v3-base-zeroshot-v1")

Device set to use mps:0


In [53]:
labels = ["legal", "scientific", "financial", "contract", "email", "invoice", "contract"]
result = classifier(full_text, candidate_labels=labels)

In [59]:
labels = [
    "abstract",
    "introduction",
    "background information",
    "research hypothesis",
    "methods",
    "experimental results",
    "data analysis",
    "discussion",
    "conclusion",
    "references",
    "limitations",
    "future work",
    "acknowledgments",
    "author information",
    "funding information",
    "ethical statement",
    "table",
    "figure",
    "mathematical formula",
    "contact information"
]

result = classifier(full_text, candidate_labels=labels)
{result['labels'][i]: result['scores'][i] for i in range(len(result['labels'])) if result['scores'][i] > 0.1}

{'abstract': 0.2148286998271942,
 'methods': 0.21315690875053406,
 'acknowledgments': 0.1387038379907608,
 'experimental results': 0.1308101862668991,
 'author information': 0.12777996063232422}

In [60]:
def classify_text(labels:list[str],text:str) -> Dict[str, float]:
    result = classifier(text, candidate_labels=labels)
    return {result['labels'][i]: result['scores'][i] for i in range(len(result['labels'])) if result['scores'][i] > 0.1}

In [62]:
classify_text(labels,all_chunks_bm25[file_name][0].page_content)

{'contact information': 0.2500768005847931,
 'references': 0.2481307089328766,
 'author information': 0.15741372108459473,
 'acknowledgments': 0.1371646672487259,
 'abstract': 0.10368910431861877}

In [81]:
legal_labels_10 = [
    "parties and people involved",
    "definitions and interpretations",
    "key dates and periods",
    "terms and conditions",
    "obligations and responsibilities",
    "rights and remedies",
    "payment and financial terms",
    "confidentiality and privacy",
    "dispute resolution and governing law",
    "signature and execution"
]

classify_text(legal_labels_10,all_chunks_bm25['rental-agreement_cleaned.txt'][2].page_content)

{'obligations and responsibilities': 0.5841822624206543,
 'parties and people involved': 0.213653102517128,
 'payment and financial terms': 0.11051750928163528}

In [109]:
manual_labels_10 = [
    "product overview and description",
    "technical specifications",
    "instructions and warnings",
    "setup and installation",
    "numbered points",
    "maintenance and troubleshooting",
    "parts and components list",
    "figures or diagrams",
    "tables",
    "warranty and service information",
    "contact details and customer support"
]
classify_text(manual_labels_10,all_chunks_bm25['aldi-document-1_cleaned.txt'][1].page_content)

{'numbered points': 0.6786946654319763}

In [110]:
all_chunks_bm25.keys()

dict_keys(['motherboard-manual_cleaned.txt', 'aldi-document-2_cleaned.txt', 'attention-is-all-you-need_cleaned.txt', 'financial-report_cleaned.txt', 'rental-agreement_cleaned.txt', 'aldi-document-1_cleaned.txt'])

In [117]:
financial_labels_10 = [
    "executive summary or management commentary",
    "balance sheet",
    "income statement",
    "cash flow statement",
    "statement of changes in equity",
    "financial ratios and key metrics",
    "notes to the financial statements",
    "auditor’s report or certification",
    "figures, charts, or financial tables",
    "contact details and investor relations"
]
classify_text(financial_labels_10,all_chunks_bm25['financial-report_cleaned.txt'][16].page_content)

{'notes to the financial statements': 0.9449139833450317}

In [118]:
print(all_chunks_bm25['financial-report_cleaned.txt'][16].page_content)

# 4. Analysis of financial position, operating results and cash flow by management Consolidated financial statements of Nintendo are prepared in accordance with accounting standards generally accepted in Japan. In preparing such statements, estimates that may affect the value of assets, liabilities, revenue and expenses are made based on the accounting procedures selected and adopted by management. Management makes appropriate assumptions based on factors such as past performance and the likelihood of future occurrences to make reasonable estimates. However, due to inherent uncertainties, the actual results may differ from these estimates. Significant accounting estimates and assumptions adopted in the consolidated financial statements of Nintendo are detailed in the section “V. Financial Information, Consolidated financial statements, etc., Notes to Consolidated Financial Statements (Significant accounting estimates).”


In [124]:
from sklearn.datasets import fetch_california_housing, load_diabetes
data = fetch_california_housing()
data = load_diabetes()
X, y = data.data, data.target

In [125]:
pd.DataFrame(X, columns=data.feature_names).head()

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641
